<h3/>
<h1 align="center">Universidad de Buenos Aires (FIUBA)</h1>
<h2 align="center">Especialización en Inteligencia Artificial - CEIA - FIUBA</h2>
<h2 align="center">Procesamiento del Lenguaje Natural I</h2>
<h3 align="center">Desafío 2 - Custom embeddings con Gensim</h3>

<p align="left">
Alumno:
  <ul>
    <li>Nicolás Lastra</li>
  </ul>
</p>

# Planteo del problema y fuente de datos

Se entrenan embeddings de palabras propios con **Word2Vec** (Gensim) sobre un corpus en español: **_Don Quijote de la Mancha_** (Miguel de Cervantes), en la versión de texto plano de [Project Gutenberg](https://www.gutenberg.org/ebooks/2000). Los vectores resultantes reflejan cómo se usan las palabras en ese texto en particular.

> **Cada experimento debe estar acompañado de una explicación o interpretación de lo observado**, incluido el sustento de las decisiones tomadas (por ejemplo, cambios de hiperparámetros). La notebook tiene que poder correrse de inicio a fin sin errores.

## Consigna

1. **Embeddings propios.** Crear vectores con Gensim a partir de un corpus propio.
2. **Términos similares.** Elegir términos de interés y buscar los más y los menos similares.
3. **Reducción de dimensionalidad.** Llevar los embeddings a 2 dimensiones y graficarlos, eligiendo `MAX_WORDS` de forma que la visualización sea legible.
4. **Grupos de palabras.** Inspeccionar el gráfico, buscar pequeños grupos de palabras, interpretarlos y sacar conclusiones, acompañados con capturas.

---
## 0. Librerías y datos

In [1]:
from pathlib import Path
import re
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models.callbacks import CallbackAny2Vec
from gensim.models import Word2Vec
from collections import Counter

# Modelo de NLTK que detecta dónde termina cada oración
nltk.download("punkt_tab")   

# Definimos semilla para reproducibilidad
SEED = 42


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lastr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
ruta = Path("../docs/data/quijote.txt")

# El corpus está versionado en docs/data/. Se descargó de Project Gutenberg con:
# import urllib.request
# URL = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
# urllib.request.urlretrieve(URL, ruta)

if not ruta.exists():
    raise FileNotFoundError(f"No se encontró {ruta}. Descargarlo con las líneas comentadas de arriba.")

# Leemos documento
texto = ruta.read_text(encoding="utf-8")

# Se recorta encabezado y licencia de Gutenberg
inicio = texto.find("*** START OF")
fin = texto.find("*** END OF")
texto = texto[texto.find("\n", inicio) + 1 : fin]
print(f"{len(texto):,} caracteres")


2,110,713 caracteres


---
## 1. Preprocesamiento y entrenamiento de Word2Vec

In [3]:
# Preprocesamiento: separar en oraciones y tokenizar

# Tokenizador propio, pasa a minúsculas deja solo palabras (incluye tildes, ü y ñ).
# Descarta puntuación, números y los guiones de diálogo.
def tokenizar(oracion):
    return re.findall(r"[a-záéíóúüñ]+", oracion.lower()) #Pasamos a minusculas cada oracion y luego tokenizamos con el patron elegido.

oraciones_texto = sent_tokenize(texto, language="spanish") # Utilizamos el tokenizador de NLTK declarando el idioma, extraemos oraciones.
oraciones = [tokenizar(o) for o in oraciones_texto] # Aplicamos tokenizador con el patron propio, tokenizamos cada oración.
oraciones = [o for o in oraciones if len(o) >= 3]   # Descartamos oraciones de 1 o 2 palabras

n_tokens = sum(len(o) for o in oraciones)
print(f"Oraciones: {len(oraciones):,}")
print(f"Tokens:    {n_tokens:,}")
print(oraciones[0])
print(oraciones[1])
print(oraciones[2])

Oraciones: 9,260
Tokens:    383,089
['el', 'ingenioso', 'hidalgo', 'don', 'quijote', 'de', 'la', 'mancha', 'por', 'miguel', 'de', 'cervantes', 'saavedra', 'el', 'ingenioso', 'hidalgo', 'don', 'quijote', 'de', 'la', 'mancha', 'tasa', 'testimonio', 'de', 'las', 'erratas', 'el', 'rey', 'al', 'duque', 'de', 'béjar', 'prólogo', 'al', 'libro', 'de', 'don', 'quijote', 'de', 'la', 'mancha', 'que', 'trata', 'de', 'la', 'condición', 'y', 'ejercicio', 'del', 'famoso', 'hidalgo', 'don', 'quijote', 'de', 'la', 'mancha', 'que', 'trata', 'de', 'la', 'primera', 'salida', 'que', 'de', 'su', 'tierra', 'hizo', 'el', 'ingenioso', 'don', 'quijote', 'donde', 'se', 'cuenta', 'la', 'graciosa', 'manera', 'que', 'tuvo', 'don', 'quijote', 'en', 'armarse', 'caballero', 'de', 'lo', 'que', 'le', 'sucedió', 'a', 'nuestro', 'caballero', 'cuando', 'salió', 'de', 'la', 'venta', 'donde', 'se', 'prosigue', 'la', 'narración', 'de', 'la', 'desgracia', 'de', 'nuestro', 'caballero', 'del', 'donoso', 'y', 'grande', 'escrutini

In [4]:
# Análisis de la distribución de frecuencias del corpus
#
# Se cuentan dos cosas a distinguir:
#   - PALABRAS DISTINTAS: cada palabra cuenta una vez, sin importar cuántas veces aparezca
#   - TOKENS: cada aparición en el texto cuenta, independientemente de si se repite o no.

# Cantidad de apariciones de cada palabra. Las claves son las palabras distintas, los valores sus apariciones.
frecuencias = Counter(palabra for oracion in oraciones for palabra in oracion)

# TOKENS: total de apariciones de palabras en el corpus, con repeticiones
total_tokens = sum(frecuencias.values())

# TOKENS: apariciones de las palabras con 5 o más repeticiones (las que entran al vocabulario con min_count=5)
cubiertos = sum(f for f in frecuencias.values() if f >= 5)

# PALABRAS DISTINTAS: cada palabra que cumple suma 1
vocabulario = sum(f >= 5 for f in frecuencias.values())
aparecen_una_vez = sum(f == 1 for f in frecuencias.values())
aparecen_menos_de_5 = sum(f < 5 for f in frecuencias.values())

print("Tokens")
print(f"  Total en el corpus:                           {total_tokens:,}")
print(f"  Cubiertos por el vocabulario (min_count>=5):  {cubiertos:,} ({cubiertos / total_tokens:.1%})")
print()
print("Palabras distintas")
print(f"  Total en el corpus:                           {len(frecuencias):,}")
print(f"  Aparecen una sola vez:                        {aparecen_una_vez:,}")
print(f"  Aparecen menos de 5 veces (se descartan):     {aparecen_menos_de_5:,}")
print(f"  Vocabulario con min_count=5:                  {vocabulario:,}")

Tokens
  Total en el corpus:                           383,089
  Cubiertos por el vocabulario (min_count>=5):  355,114 (92.7%)

Palabras distintas
  Total en el corpus:                           22,870
  Aparecen una sola vez:                        11,138
  Aparecen menos de 5 veces (se descartan):     17,552
  Vocabulario con min_count=5:                  5,318


In [5]:
# Utilizamos la clase base de CallbackAny2Vec para informar el loss por época.    
# Gensim acumula el loss en un float32 que pierde precisión cuando el valor crece
# (mostraba una caída falsa desde la época 25). Se resetea al final de cada época
# para obtener el loss de esa época sola.

class callback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        print(f"Época {self.epoch:2d} - loss: {loss:,.0f}")
        model.running_training_loss = 0.0   # Reset del acumulado interno de Gensim
        self.epoch += 1

In [6]:
# Modelo Word2Vec: vocabulario y entrenamiento

modelo_sg = Word2Vec(
    vector_size=100,   # Dimension de cada embedding.
    window=5,          # Ventana de contexto, 5 palabras a cada lado, las oraciones del Quijote son largas.
    min_count=5,       # Palabras con menos de 5 apariciones quedan fuera del vocabulario ya que no tienen contexto suficiente.
    sg=1,              # Utiliza skip-gram, mejor que CBOW para corpus chicos y palabras poco frecuentes.
    negative=10,       # Por cada par real se comparan 10 palabras al azar como ejemplos negativos.
    workers=1,         # Utiliza un solo hilo de la CPU para que el resultado sea reproducible.
    seed=SEED,         # Semilla, junto con workers fijan la reproducibilidad.
)

modelo_sg.build_vocab(oraciones)
print(f"Palabras distintas en el vocabulario (min_count=5): {len(modelo_sg.wv):,}")

Palabras distintas en el vocabulario (min_count=5): 5,318


In [7]:
# Entrenamiento: 30 pasadas completas por el corpus, imprimiendo el loss de cada época
modelo_sg.train(
    oraciones,
    total_examples=modelo_sg.corpus_count,   # cantidad de oraciones, la calculó con build_vocab
    epochs=30,                                 # pasadas completas por el corpus
    compute_loss=True,                         # necesario para que el callback tenga un loss para mostrar
    callbacks=[callback()],
)

Época  0 - loss: 4,215,029
Época  1 - loss: 3,837,931
Época  2 - loss: 3,785,520
Época  3 - loss: 3,757,006
Época  4 - loss: 3,722,410
Época  5 - loss: 3,687,572
Época  6 - loss: 3,673,818
Época  7 - loss: 3,652,191
Época  8 - loss: 3,637,644
Época  9 - loss: 3,621,591
Época 10 - loss: 3,604,104
Época 11 - loss: 3,598,559
Época 12 - loss: 3,596,131
Época 13 - loss: 3,589,518
Época 14 - loss: 3,575,555
Época 15 - loss: 3,570,028
Época 16 - loss: 3,556,740
Época 17 - loss: 3,565,764
Época 18 - loss: 3,552,385
Época 19 - loss: 3,544,636
Época 20 - loss: 3,541,577
Época 21 - loss: 3,555,216
Época 22 - loss: 3,545,584
Época 23 - loss: 3,543,840
Época 24 - loss: 3,546,727
Época 25 - loss: 3,557,315
Época 26 - loss: 3,537,146
Época 27 - loss: 3,542,880
Época 28 - loss: 3,554,466
Época 29 - loss: 3,552,744


(7164920, 11492670)

In [8]:
#Realizamos el mismo seteo pero utilizando la arquitectura CBOW
modelo_cbow = Word2Vec(
    vector_size=100, 
    window=5, 
    min_count=5,
    sg=0,          
    negative=10,
    workers=1,
    seed=SEED,
)

modelo_cbow.build_vocab(oraciones)
print(f"Palabras distintas en el vocabulario (min_count=5): {len(modelo_cbow.wv):,}")

Palabras distintas en el vocabulario (min_count=5): 5,318


In [9]:
# Entrenamiento: 30 pasadas completas por el corpus, imprimiendo el loss de cada época
modelo_cbow.train(
    oraciones,
    total_examples=modelo_cbow.corpus_count,
    epochs=30,
    compute_loss=True,
    callbacks=[callback()],
)

Época  0 - loss: 934,561
Época  1 - loss: 797,514
Época  2 - loss: 757,027
Época  3 - loss: 727,018
Época  4 - loss: 703,400
Época  5 - loss: 685,859
Época  6 - loss: 672,878
Época  7 - loss: 660,064
Época  8 - loss: 648,365
Época  9 - loss: 639,606
Época 10 - loss: 629,192
Época 11 - loss: 622,864
Época 12 - loss: 615,456
Época 13 - loss: 609,376
Época 14 - loss: 603,040
Época 15 - loss: 596,510
Época 16 - loss: 590,545
Época 17 - loss: 586,772
Época 18 - loss: 581,149
Época 19 - loss: 576,702
Época 20 - loss: 573,516
Época 21 - loss: 572,325
Época 22 - loss: 566,865
Época 23 - loss: 564,946
Época 24 - loss: 561,735
Época 25 - loss: 558,841
Época 26 - loss: 555,845
Época 27 - loss: 554,307
Época 28 - loss: 553,836
Época 29 - loss: 552,062


(7164920, 11492670)

**Análisis:**

**Corpus.** Se usó *Don Quijote de la Mancha* en la versión de texto plano de Project Gutenberg, sin el encabezado ni la licencia en inglés del archivo original (quedan 2.110.713 caracteres). Recortarlos evita que el modelo aprenda vocabulario ajeno al libro ("license", "copyright", "foundation").

**Preprocesamiento.** Se hizo en dos niveles:

1. **Oraciones**, con `sent_tokenize` de NLTK (modelo Punkt para español), que distingue los puntos finales de los de abreviaturas. Word2Vec necesita el texto separado en oraciones porque la ventana de contexto se mueve dentro de cada una, y así no relaciona la última palabra de una oración con la primera de la siguiente.
2. **Palabras**, con un tokenizador propio basado en una expresión regular (`[a-záéíóúüñ]+`) aplicada sobre el texto en minúsculas. Se eligió en lugar de `word_tokenize`, que deja los signos `¿` y `¡` pegados a la palabra (`'¿Qué'`): con el filtro `isalpha()` del Desafío 1, esas palabras se habrían descartado enteras.

Se descartaron las oraciones de menos de 3 palabras (títulos sueltos y réplicas cortas de diálogo). Quedaron **9.260 oraciones y 383.089 tokens**.

Las decisiones de preprocesamiento son distintas a las del Desafío 1, porque cambia lo que se hace después:

| Decisión | Desafío 1 (TF-IDF) | Desafío 2 (Word2Vec) | Motivo |
|---|---|---|---|
| Stopwords | Se eliminaron | **Se mantuvieron** | Word2Vec aprende del contexto, eliminarlas cambiaría las vecinas de cada palabra dentro de la ventana |
| Lematización | Sí | **No** | Se prefiere que el modelo aprenda por sí mismo la relación entre variantes ("dinero" y "dineros" quedan entre sus vecinos mutuos) |
| Tildes | — | **Se mantuvieron** | En español distinguen palabras distintas: *el/él* (8.267 / 1.282 apariciones), *si/sí* (1.968 / 346), *mas/más* (240 / 2.055; Cervantes usa "mas" como "pero") |

**Limitación observada.** `sent_tokenize` solo corta en `.`, `?` y `!`, y Cervantes encadena frases con `;` y `:`, por lo que hay "oraciones" de cientos de palabras (la más larga tiene 2.442). Además, la portada y el índice del libro quedaron unidos en la primera oración, porque esas líneas no tienen punto. No afecta demasiado al entrenamiento, porque la ventana de contexto limita igual cuántas palabras se relacionan entre sí.

**Vocabulario.**

| | Palabras distintas | Tokens |
|---|---|---|
| Total en el corpus | 22.870 | 383.089 |
| Aparecen una sola vez | 11.138 (49 %) | — |
| Descartadas (menos de 5 apariciones) | 17.552 (77 %) | 27.975 (7,3 %) |
| **Vocabulario con `min_count=5`** | **5.318 (23 %)** | **355.114 (92,7 %)** |

Con `min_count=5` el vocabulario se reduce a menos de un cuarto, pero conserva el **92,7 % del texto**. Las palabras descartadas aparecen tan pocas veces que el modelo no tendría contextos suficientes para aprender un vector confiable, casi la mitad del vocabulario aparece una sola vez.

**Hiperparámetros.**

| Parámetro | Valor | Justificación |
|---|---|---|
| `vector_size` | 100 | Con ~380 mil tokens no hay datos suficientes para aprender bien 300 dimensiones
| `window` | 5 | Una ventana amplia captura relaciones de tema (personajes, escenas) más que de función gramatical |
| `min_count` | 5 | Ver tabla de vocabulario |
| `sg` | 1 (skip-gram) | Funciona mejor que CBOW en corpus chicos y con palabras poco frecuentes (se valida en el punto 2) |
| `negative` | 10 | Valor recomendado para corpus chicos (entre 5 y 20) |
| `workers` y `seed` | 1 y 42 | Con un solo hilo y semilla fija, el entrenamiento es reproducible |

**Entrenamiento.** En la primera corrida, el loss mostró una caída de 2,5 millones a ~118 mil en la época 25. No era una mejora real, Gensim acumula el loss en un `float32`, que pierde precisión cuando el acumulado supera las decenas de millones, y las restas entre épocas daban valores falsos. Se corrigió reseteando el acumulado al final de cada época en el callback.

Con la corrección, el loss de skip-gram baja de 4.215.029 a 3.837.931 en la primera época (−9 %), sigue bajando lentamente hasta la época 15 (~3,57 millones) y luego oscila en torno a 3,55 millones con variaciones menores al 0,5 %. El modelo convergió, y **30 épocas son suficientes**.

**Skip-gram frente a CBOW.** Se entrenó un segundo modelo con los mismos hiperparámetros y `sg=0`. CBOW termina con un loss de ~552 mil, contra ~3,55 millones de skip-gram, y es unas 4 veces más rápido. **Esos valores no son comparables**, el loss es una suma sobre los ejemplos de la época, y CBOW hace una predicción por palabra central, mientras que skip-gram hace una por cada par (central, contexto), unas 6 veces más ejemplos. Lo comparable es la forma de las curvas, ambas bajan y se estabilizan. La calidad de cada modelo se compara a partir de los términos similares en el punto 2 de esta notebook.

---
## 2. Términos más y menos similares

In [ ]:
# Términos de interés elegidos a mano

In [ ]:
# Más similares (most_similar positive)

In [ ]:
# Menos similares (most_similar negative)

**Análisis:**

_Completar._

---
## 3. Reducción de dimensionalidad y visualización

In [ ]:
# Reducción a 2 dimensiones

In [ ]:
MAX_WORDS = 200

# Gráfico de los embeddings proyectados

**Análisis:**

_Completar._

---
## 4. Grupos de palabras

_Grupos identificados en el gráfico, con capturas en `docs/img/`:_

<!-- ![Grupo 1](../docs/img/desafio2_grupo1.png) -->

**Análisis:**

_Completar._

---
## Conclusiones

_Completar._